![Banner](img/AI_Special_Program_Banner.jpg)

# Introduction to LLMs - Exercise, Task 1: Use RAG to answer questions
---

[back to the exercise notebook](3.4_LC_Ex.ipynb)

### $\color{teal}{\textsf{Task 1: Use RAG to answer questions}}$

Use your local Ollama LLM server to answer the following questions:

* What does self-attention do in the context of transformers?
* What are positional encodings and why are they needed?

**Hints**:
* The answers may be found in the original [transformer paper](https://arxiv.org/abs/1706.03762) and
* you must find a suitable document loader (see the link in the [RAG notebook](3.4_4_LC_RAG_Ollama.ipynb)).

---

We rebuild the pipeline of the [RAG notebook](3.4_4_LC_RAG_Ollama.ipynb), but exchange the *knowledge
source*: instead of a HuggingFace model card we index *Attention Is All You Need* (Vaswani et al., 2017).
Both questions are answered in that paper, so a working retriever should be able to find them - and,
just as importantly, the answers should then be *grounded in the paper* rather than in whatever the
model remembers from its training data. The last section checks exactly that by asking the same
questions once **without** any retrieval.

## Imports etc.

In [1]:
import os

# in order to avoid a notification about a missing USER_AGENT;
# must be done before the langchain imports
os.environ["USER_AGENT"] = "I_dont_care"

In [2]:
import datetime as dt
import textwrap

from langchain_ollama import ChatOllama, OllamaEmbeddings

# we can use this to render the LLM responses in Markdown ...
from IPython.display import display, Markdown

import warnings
warnings.filterwarnings('ignore')

In [3]:
from pt_utils import print_execution_details

# Documentation
show_libraries = False
print_execution_details(show_libraries)

EXECUTION DETAILS

🕒 Run Start Time: 2026-07-30 17:58:16
💻 Operating Sys:  Windows-11-10.0.26200-SP0
🐍 Python Version: 3.12.2

ℹ️  Library versions hidden (show_libs=False).



## Configuration

As in the material notebook, everything that may have to be adapted to a different machine or a
different model lives in this one cell. Note that the models here are deliberately small: this notebook
runs on a laptop CPU, and for questions whose answers are handed to the model in the context anyway, a
small model is perfectly sufficient (that is, after all, the whole point of RAG).

In [4]:
# --- Where the models are served -------------------------------------------
OLLAMA_BASE_URL = "http://localhost:11434"

# --- Language model --------------------------------------------------------
os_model = "gemma3"          # the default tag of gemma3 IS the 4B variant

TEMPERATURE = 0.7
TOP_P = 0.95
MAX_NEW_TOKENS = 1024
NUM_CTX = 8192               # must comfortably hold the retrieved chunks + the question!
SEED = 42

# --- Embedding model -------------------------------------------------------
EMBEDDING_MODEL = "embeddinggemma"       # 300M parameters, 768 dimensions, multilingual

# EmbeddingGemma is trained to be used with task prefixes (see the RAG notebook)
APPLY_TASK_PREFIXES = True
QUERY_PREFIX = "task: search result | query: "
DOCUMENT_PREFIX = "title: none | text: "

# --- The document we want to ask about --------------------------------------
ARXIV_ID = "1706.03762"      # Vaswani et al., "Attention Is All You Need"

## Is the server up and are the models there?

The very first thing that goes wrong in this setup is a model that has not been pulled, and the error
message one gets in that case is not particularly helpful. So we check first.

In [5]:
import ollama

client = ollama.Client(host=OLLAMA_BASE_URL)


def list_local_models(cl):
    """Return the names of all models available on the server, across client versions."""
    result = cl.list()
    entries = result.models if hasattr(result, "models") else result["models"]
    names = []
    for e in entries:
        name = getattr(e, "model", None) or getattr(e, "name", None)
        if name is None:                      # old client: plain dicts
            name = e.get("model") or e.get("name")
        names.append(name)
    return sorted(names)


available = list_local_models(client)
print(f'Ollama at {OLLAMA_BASE_URL} is up. Models available:\n{"=" * 20}\n')
for name in available:
    needed = name.split(":")[0] in (os_model.split(":")[0], EMBEDDING_MODEL.split(":")[0])
    print(f'  - {name}{"   <-- used here" if needed else ""}')

for wanted in (os_model, EMBEDDING_MODEL):
    if not any(a == wanted or a.startswith(wanted + ":") for a in available):
        print(f"\n!! '{wanted}' is missing - fetch it with:  ollama pull {wanted}")

Ollama at http://localhost:11434 is up. Models available:

  - deepseek-r1:1.5b
  - embeddinggemma:latest   <-- used here
  - gemma2:2b
  - gemma3:latest   <-- used here
  - qwen2.5:0.5b
  - qwen2.5:1.5b


## Initializing the LLM

This is the only place where the language model shows up at all - everything below is
provider-agnostic LangChain code.

In [6]:
llm_kwargs = dict(
    model=os_model,
    base_url=OLLAMA_BASE_URL,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    num_predict=MAX_NEW_TOKENS,
    num_ctx=NUM_CTX,
    seed=SEED,
)

# Reasoning models emit their "thinking" before the actual answer, which we do not want
# in the RAG output. Only pass the parameter if this version of langchain-ollama knows it.
if "reasoning" in ChatOllama.model_fields:
    llm_kwargs["reasoning"] = False

llm = ChatOllama(**llm_kwargs)

## Document Loader

This is the part of the pipeline the task is really about. The RAG notebook used `WebBaseLoader`, which
fetches an HTML page and strips the tags - and that is exactly what does *not* work here: the source is
a **PDF** on arXiv, and `WebBaseLoader` would hand us the raw bytes of the PDF (or the HTML of the
abstract page, which contains only the summary, not the answers).

Looking through the [list of document loaders](https://python.langchain.com/docs/integrations/document_loaders/)
linked in the RAG notebook, two of them fit, and they differ in a way that matters:

* `PyPDFLoader` - the generic PDF loader. It also accepts a URL, and it returns **one document per
  page**, so the page number ends up in the metadata of every chunk.
* `ArxivLoader` - the specialised one: give it an arXiv ID and it fetches the paper together with its
  bibliographic metadata (title, authors, publication date) - but as **one single document**, so the
  page information is lost.

We index the `PyPDFLoader` version, because for a question about a paper we want to be able to say
*where* in the paper the answer came from. The next cell shows the alternative for comparison.

In [7]:
from langchain_community.document_loaders import PyPDFLoader

PDF_URL = f"https://arxiv.org/pdf/{ARXIV_ID}"

data = PyPDFLoader(PDF_URL).load()      # one document per page
loader_used = "PyPDFLoader"

print(f'Loaded with {loader_used}: {len(data)} document(s) (= pages), '
      f'{sum(len(d.page_content) for d in data):,} characters in total\n')
print(f'Metadata of the first document:\n{"=" * 20}')
for key, value in data[0].metadata.items():
    print(f'  {key}: {str(value)[:100]}')

Loaded with PyPDFLoader: 15 document(s) (= pages), 39,597 characters in total

Metadata of the first document:
  producer: pdfTeX-1.40.25
  creator: LaTeX with hyperref
  creationdate: 2024-04-10T21:11:43+00:00
  author: 
  keywords: 
  moddate: 2024-04-10T21:11:43+00:00
  ptex.fullbanner: This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5
  subject: 
  title: 
  trapped: /False
  source: https://arxiv.org/pdf/1706.03762
  total_pages: 15
  page: 0
  page_label: 1


For comparison, the specialised loader. It gives us the bibliographic metadata that `PyPDFLoader`
cannot know - but a single document without page numbers.

**Version note:** `ArxivLoader` needs `arxiv < 3`. With `arxiv` 4.x, `langchain-community` still calls
`Search.results()`, which no longer exists, and the loader raises an `AttributeError` - a good example
of the versioning trouble the first material notebook warns about.

In [8]:
try:
    from langchain_community.document_loaders import ArxivLoader

    arxiv_data = ArxivLoader(query=ARXIV_ID, load_max_docs=1).load()
    print(f'ArxivLoader : {len(arxiv_data)} document(s), '
          f'{sum(len(d.page_content) for d in arxiv_data):,} characters')
    print(f'  metadata  : {list(arxiv_data[0].metadata)}')
    print(f'  title     : {arxiv_data[0].metadata.get("Title")}')
    print(f'  published : {arxiv_data[0].metadata.get("Published")}')
except Exception as err:
    print(f'ArxivLoader not usable here -> {type(err).__name__}: {err}')

print(f'\nPyPDFLoader : {len(data)} document(s), '
      f'{sum(len(d.page_content) for d in data):,} characters')
print(f'  metadata  : {list(data[0].metadata)}')
print(f'\n-> practically the same text; we keep the PDF loader for its page numbers.')

ArxivLoader : 1 document(s), 39,498 characters
  metadata  : ['Published', 'Title', 'Authors', 'Summary']
  title     : Attention Is All You Need
  published : 2023-08-02

PyPDFLoader : 15 document(s), 39,597 characters
  metadata  : ['producer', 'creator', 'creationdate', 'author', 'keywords', 'moddate', 'ptex.fullbanner', 'subject', 'title', 'trapped', 'source', 'total_pages', 'page', 'page_label']

-> practically the same text; we keep the PDF loader for its page numbers.


Let us convince ourselves that the text really did arrive - the paragraph on scaled dot-product
attention should be in there somewhere:

In [9]:
full_text = "\n".join(d.page_content for d in data)
pos = full_text.find("Scaled Dot-Product Attention")
print(textwrap.fill(full_text[pos:pos + 700], width=100))

Scaled Dot-Product Attention  Multi-Head Attention Figure 2: (left) Scaled Dot-Product Attention.
(right) Multi-Head Attention consists of several attention layers running in parallel. of the
values, where the weight assigned to each value is computed by a compatibility function of the query
with the corresponding key. 3.2.1 Scaled Dot-Product Attention We call our particular attention
"Scaled Dot-Product Attention" (Figure 2). The input consists of queries and keys of dimension dk,
and values of dimension dv. We compute the dot products of the query with all keys, divide each by
√dk, and apply a softmax function to obtain the weights on the values. In practice, we compute the
attention func


## Text Splitter

Same splitter as in the material notebook. One thing does change, though: a paper is a much longer and
much denser document than a model card, so we allow a small `chunk_overlap`. With an overlap of `0` a
sentence that happens to straddle a chunk boundary is torn apart and is then hard to retrieve - and in a
paper the sentence that *defines* something is precisely the one we are after.

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", "."],
    chunk_size=512,
    chunk_overlap=64,          # keep sentences that straddle a boundary retrievable
    length_function=len,
    is_separator_regex=False,
)

docs = text_splitter.split_documents(data)
print(f'The paper was split into {len(docs)} chunks '
      f'(mean length: {sum(len(d.page_content) for d in docs) / len(docs):.0f} characters)')

The paper was split into 92 chunks (mean length: 441 characters)


## Text Embedding Model and Task Prefixes

Unchanged from the material notebook: `embeddinggemma` produces the vectors, and the small wrapper
prepends the task prefixes the model was trained with.

In [11]:
embeddings_model = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url=OLLAMA_BASE_URL)
embedding_tag = EMBEDDING_MODEL.replace(":", "_")

print(f'Embedding model in use: {EMBEDDING_MODEL} '
      f'(dimension: {len(embeddings_model.embed_query("test"))})')

Embedding model in use: embeddinggemma (dimension: 768)


In [12]:
from langchain_core.embeddings import Embeddings


class PrefixedEmbeddings(Embeddings):
    """Prepend task instructions, so queries and documents are embedded as what they are."""

    def __init__(self, inner, query_prefix, document_prefix):
        self.inner = inner
        self.query_prefix = query_prefix
        self.document_prefix = document_prefix

    def embed_documents(self, texts):
        return self.inner.embed_documents([self.document_prefix + t for t in texts])

    def embed_query(self, text):
        return self.inner.embed_query(self.query_prefix + text)


if APPLY_TASK_PREFIXES:
    embeddings_model = PrefixedEmbeddings(embeddings_model, QUERY_PREFIX, DOCUMENT_PREFIX)
    print("Task prefixes are applied automatically.")

Task prefixes are applied automatically.


## Vector Store

Building the index means embedding every single chunk once, so this is the expensive cell of the
notebook - and the reason we save the index to disk afterwards.

In [13]:
from langchain_community.vectorstores import FAISS

start = dt.datetime.now()
db = FAISS.from_documents(docs, embeddings_model)
print(f'Embedding {len(docs)} chunks took {dt.datetime.now() - start}')

index_dir = f"faiss_index_transformer_{embedding_tag}"
db.save_local(index_dir)          # an index is only ever valid with the model that produced it
print(f'Index persisted to "{index_dir}"')

Embedding 92 chunks took 0:00:21.117852
Index persisted to "faiss_index_transformer_embeddinggemma"


## Retriever and Retrieval Chain

Note the addition to the prompt template: besides restricting the model to the context, we tell it to
say so when the context does not contain the answer. Without that sentence the model happily falls back
on its own memory, which is exactly the failure mode RAG is supposed to remove.

In [14]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

K = 5                                        # a paper needs a bit more context than a model card
retriever = db.as_retriever(search_kwargs={"k": K})


# helper function to format the retrieved chunks/documents
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context, which is taken from a research paper.
If the context does not contain the answer, say so instead of guessing.

{context}

Question: {question}
""")

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

A small helper so that both questions can be asked in the same way - and so that we can always see
*what the model was actually given*. In RAG, a disappointing answer is far more often a retrieval
problem than a generation problem.

In [15]:
rag_secs = {}          # question -> seconds, so we can compare the cost later


def ask_rag(question, show_context=True, preview=200):
    """Run the RAG chain and (optionally) show the chunks the model received."""
    if show_context:
        retrieved = retriever.invoke(question)
        print(f'{len(retrieved)} chunks were handed to the model:\n')
        for i, d in enumerate(retrieved, start=1):
            page = d.metadata.get("page", "-")
            print(f'--- chunk {i} (page {page}) ' + "-" * 40)
            print(textwrap.fill(d.page_content[:preview].strip(), width=100))
            print()

    start = dt.datetime.now()
    answer = rag_chain.invoke(question)
    duration = dt.datetime.now() - start
    rag_secs[question] = duration.total_seconds()

    print(f'This is {os_model}\'s answer to ** {question} ** (in {duration}):\n{"+-" * 25}\n')
    display(Markdown(answer))
    print(f'\n{"+-" * 25}')
    return answer

## $\color{teal}{\textsf{Question 1: What does self-attention do in the context of transformers?}}$

In [16]:
q1 = "What does self-attention do in the context of transformers?"
answer_q1 = ask_rag(q1)

5 chunks were handed to the model:

--- chunk 1 (page 1) ----------------------------------------
described in section 3.2. Self-attention, sometimes called intra-attention is an attention mechanism
relating different positions of a single sequence in order to compute a representation of the seque

--- chunk 2 (page 1) ----------------------------------------
End-to-end memory networks are based on a recurrent attention mechanism instead of sequence- aligned
recurrence and have been shown to perform well on simple-language question answering and language m

--- chunk 3 (page 4) ----------------------------------------
and queries come from the same place, in this case, the output of the previous layer in the encoder.
Each position in the encoder can attend to all positions in the previous layer of the encoder. • Si

--- chunk 4 (page 5) ----------------------------------------
layer in a typical sequence transduction encoder or decoder. Motivating our use of self-attention we
consider 

This is gemma3's answer to ** What does self-attention do in the context of transformers? ** (in 0:00:31.076734):
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-



Self-attention relates different positions of a single sequence to compute a representation of the sequence. Specifically, each position in the encoder can attend to all positions in the previous layer of the encoder, and similarly, self-attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder up to and including that position.


+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-


## $\color{teal}{\textsf{Question 2: What are positional encodings and why are they needed?}}$

In [17]:
q2 = "What are positional encodings and why are they needed?"
answer_q2 = ask_rag(q2)

5 chunks were handed to the model:

--- chunk 1 (page 5) ----------------------------------------
bottoms of the encoder and decoder stacks. The positional encodings have the same dimension dmodel
as the embeddings, so that the two can be summed. There are many choices of positional encodings,
lea

--- chunk 2 (page 5) ----------------------------------------
Convolutional O(k · n · d2) O(1) O(logk(n)) Self-Attention (restricted) O(r · n · d) O(1) O(n/r) 3.5
Positional Encoding Since our model contains no recurrence and no convolution, in order for the mod

--- chunk 3 (page 5) ----------------------------------------
corresponds to a sinusoid. The wavelengths form a geometric progression from 2π to 10000 · 2π. We
chose this function because we hypothesized it would allow the model to easily learn to attend by
rela

--- chunk 4 (page 2) ----------------------------------------
sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This
masking, comb

This is gemma3's answer to ** What are positional encodings and why are they needed? ** (in 0:00:24.824288):
+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-



According to the text, positional encodings are sine and cosine functions of different frequencies that correspond to a sinusoid. They are needed because the model contains no recurrence or convolution, so information about the relative or absolute position of tokens in the sequence must be injected. This allows the model to make use of the order of the sequence.


+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-+-


## Does the retrieval actually help?

The two answers above look convincing - but that is not yet evidence that the *retrieval* did anything:
the transformer paper is one of the most-cited papers in machine learning, so a model may well be able
to answer both questions from memory. To find out, we ask the very same questions again, this time
straight to the LLM with no context at all.

In [18]:
plain_prompt = ChatPromptTemplate.from_template("{question}")
plain_chain = plain_prompt | llm | StrOutputParser()

no_rag, no_rag_secs = {}, {}
for q in (q1, q2):
    start = dt.datetime.now()
    no_rag[q] = plain_chain.invoke(q)
    no_rag_secs[q] = (dt.datetime.now() - start).total_seconds()
    print(f'--- WITHOUT RAG: {q}  ({no_rag_secs[q]:.1f} s)\n')
    display(Markdown(no_rag[q]))
    print(f'\n{"=" * 100}\n')

--- WITHOUT RAG: What does self-attention do in the context of transformers?  (122.0 s)



Self-attention is a core mechanism within transformer models that allows them to understand the relationships between different parts of an input sequence (like a sentence). It's what gives transformers their remarkable ability to capture long-range dependencies and contextual nuances. Here’s a breakdown of how it works:

**1. The Problem Traditional Models Faced:**

* **Recurrent Networks (RNNs, LSTMs):**  Older models like RNNs process sequences one element at a time, maintaining a "hidden state" that summarizes the information seen so far. However, this creates challenges:
    * **Sequential Processing:** Slow and difficult to parallelize.
    * **Vanishing/Exploding Gradients:** Difficulties in learning long-range dependencies – information from early parts of the sequence can get lost as it’s processed through many steps.

**2. Self-Attention's Solution: Parallel Contextualization**

Self-attention solves these problems by allowing *every* element in a sequence to directly attend to *every other* element simultaneously. Instead of processing sequentially, it creates a contextualized representation for each word based on its entire context within the sentence.

**3. The Process – Key Steps:**

Let's break down the self-attention mechanism into three main steps:

   a. **Queries, Keys, and Values (Q, K, V):**
      * Each input token (word) is transformed into three vectors: a **Query (Q)**, a **Key (K)**, and a **Value (V)**. These are learned linear projections of the original word embedding. Think of them like this:
          * **Query:** Represents what the current word is *looking for*.  "What other words in the sentence are relevant to me?"
          * **Key:** Represents what information each word *offers*. "Here's what I have to contribute."
          * **Value:** Contains the actual information from the word that might be useful.

   b. **Calculating Attention Scores:**
      * For each token, its Query (Q) is compared to the Key (K) of every other token in the sequence using a dot product. This produces a "score" representing how much attention the current token should pay to each of the others.  Higher scores indicate more relevance.
      * `Attention Scores = Q * K^T` (where K^T is the transpose of the Key matrix)

   c. **Softmax and Weighted Sum:**
      * The attention scores are then passed through a softmax function, normalizing them into probabilities that sum to 1. This creates a probability distribution representing the importance of each word in relation to the current one.
      * Finally, these probabilities are used as weights to calculate a weighted sum of the Value (V) vectors.  This results in a single context vector for the original token – a representation that incorporates information from all other relevant words in the sentence, weighted by their attention scores.

   **Formula:** `Attention(Q, K, V) = softmax((Q * K^T) / sqrt(d_k)) * V`
      *  `d_k`: The dimensionality of the key vectors – used for scaling to prevent gradients from becoming too small or large during training.



**4. Multi-Head Attention:**

To further enhance this process, transformer models typically use **multi-head attention**. This means:

* Multiple sets of Q, K, and V matrices are learned independently.
* Each set performs the self-attention calculation ("head").
* The outputs from all heads are concatenated and linearly transformed to produce the final output.  This allows the model to capture different types of relationships between words (e.g., syntactic dependencies, semantic connections).

**5. Why is it Powerful?**

* **Parallelization:** Self-attention can be computed in parallel for all positions in the input sequence, unlike RNNs.
* **Long-Range Dependencies:** Directly connects any two tokens regardless of distance, effectively capturing long-range relationships.
* **Contextualized Representations:** Creates rich, context-aware representations of words that are superior to traditional word embeddings (like Word2Vec).


**Example (Simplified):**

Consider the sentence: "The animal didn't cross the street because it was too tired."  Self-attention would help the model understand that “it” refers to “the animal,” even though they’re separated by several words. The attention mechanism would assign a high score between “it” and “animal.”

---

**Resources for Further Learning:**

* **The Illustrated Transformer:** [http://jalammar.github.io/illustrated-transformer/](http://jalammar.github.io/illustrated-transformer/) (A fantastic visual explanation)
* **TensorFlow's Self-Attention Example:**  [https://colab.research.google.com/notebooks/tensorflow_self_attention.

--- WITHOUT RAG: What are positional encodings and why are they needed?  (111.2 s)



Okay, let's break down positional encodings – what they are and why they're crucial for neural networks, particularly in the context of sequence data like text or audio.

**1. The Problem: Neural Networks Don’t Inherently Understand Order**

* **Traditional Neural Networks (like Feedforward Nets):**  These networks treat each input independently. They don't inherently understand that "the cat sat on the mat" is different from "the mat sat on the cat." They just see a bunch of numbers representing words or features, and their connections learn relationships *between* those inputs, but not the order they appear in.
* **Sequence Data (Text, Audio, Time Series):**  In many real-world scenarios, the *order* of elements is absolutely vital to meaning. For example:
    * **Text:** "apple pie" means something entirely different from "pie apple."
    * **Audio:** The sequence of sound waves determines what we hear as a melody or speech.
    * **Time Series (stock prices, sensor readings):**  The order matters because past values influence future ones.


**2. What are Positional Encodings?**

Positional encodings are techniques used to add information about the *position* of each element in a sequence to an input before it's fed into a neural network. They essentially tell the network, "Hey, this word is at position 1, this word is at position 2, and so on."

**3. How Do They Work?  (Different Methods)**

There are several ways to create positional encodings:

* **Sine & Cosine Waves (Most Common - Used in Transformers):** This is the method used in the original Transformer architecture (which powers models like BERT).
    * It uses sine and cosine functions of different frequencies. The formula looks something like this:
       `PE(pos, 2i) = sin(pos / 10000^(2i/d_model))`
       `PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))`
    * **`pos`:** The position of the element in the sequence (e.g., 0 for the first element, 1 for the second).
    * **`i`:**  The dimension within the encoding vector.
    * **`d_model`:** The dimensionality of the embeddings (the size of the vectors representing each word/element).

    * **Why Sine & Cosine?**  These functions create a unique pattern for each position, allowing the network to learn relative positions effectively. Since sine and cosine waves oscillate at different rates, they can represent different distances in the sequence. The periodic nature helps capture relationships between elements that are far apart.

* **Learned Embeddings:**  Instead of using fixed encodings, you could train a separate embedding layer specifically for positional information. This allows the network to learn optimal encodings based on the data itself, but it can be less stable than using pre-defined functions.
* **Absolute Position Encoding:** Adding the absolute position as a vector directly to the input embeddings.
* **Relative Position Encoding:**  Instead of encoding each element's absolute position, this method encodes the *relative distance* between elements. This has become increasingly popular and is used in models like RoPE (Rotary Positional Embeddings).


**4. Why Are They Needed?**

* **Capture Order Dependency:** Without positional encodings, a neural network would be blind to the order of items in a sequence. It wouldn't understand that "dog bites man" is different from "man bites dog."
* **Improved Performance:**  Positional encodings dramatically improve the performance of models dealing with sequential data, particularly for tasks like:
    * **Machine Translation:** Knowing the word order is crucial for translating accurately.
    * **Language Modeling:** Predicting the next word in a sequence relies heavily on understanding context and position.
    * **Speech Recognition:** The order of sounds determines what we understand as speech.

**5. Key Differences from Word Embeddings**

It's important to distinguish positional encodings from *word embeddings* (like Word2Vec or GloVe).
   * **Word Embeddings:** These represent words as dense vectors, capturing semantic relationships between words (e.g., "king" is closer to "queen" than it is to "apple").  They don't capture position.
   * **Positional Encodings:** These add information about *where* a word or element appears in the sequence.

**Resources for Further Learning:**

* **The Illustrated Transformer:** [https://jalammar.github.io/illustrated-transformer/](https://jalammar.github.io/illustrated-transformer/) - Excellent visual explanation of Transformers and positional encodings.
*

To turn the impression into a measurement we need to be careful about *what* we measure.

The obvious idea - count how many technical terms of the paper appear in each answer - does not work,
and it is worth seeing why: the answer without retrieval is roughly ten times longer, and a long generic
explanation of attention mentions *query*, *key* and *value* simply because it is long. Such a count
measures verbosity, not grounding.

What we actually want to know is whether the answer was built **from the retrieved text**. So we measure
the overlap directly: of the content words in the answer, how many also occur in the chunks that were
handed to the model for that question? For the no-RAG answer we compare against the *same* chunks -
it just never got to see them.

In [19]:
import pandas as pd
import re

# a handful of function words that would inflate the overlap without meaning anything
STOP = {"that", "this", "with", "from", "have", "been", "which", "their", "there", "these",
        "those", "each", "other", "more", "than", "also", "such", "into", "when", "what",
        "were", "they", "would", "could", "because", "about", "only", "some", "them", "then",
        "here", "your", "like", "just", "very", "will", "does", "using", "used", "make"}


def content_words(text):
    return [w for w in re.findall(r"[a-z]{4,}", text.lower()) if w not in STOP]


def grounded_fraction(answer, context):
    """Share of the answer's content words that also occur in the retrieved chunks."""
    ctx = set(content_words(context))
    words = content_words(answer)
    return sum(w in ctx for w in words) / len(words) if words else 0.0


rows = []
for q, with_rag in ((q1, answer_q1), (q2, answer_q2)):
    context = format_docs(retriever.invoke(q))      # the very same chunks as above
    for label, text, secs in (("with RAG", with_rag, rag_secs[q]),
                              ("without RAG", no_rag[q], no_rag_secs[q])):
        rows.append({
            "question": q[:38] + "...",
            "variant": label,
            "characters": len(text),
            "seconds": round(secs, 1),
            "grounded": round(grounded_fraction(text, context), 2),
        })

comparison = pd.DataFrame(rows)
comparison

,question,variant,characters,seconds,grounded
0,What does self-attention do in the con...,with RAG,381,31.1,0.94
1,What does self-attention do in the con...,without RAG,4666,122.0,0.30
2,What are positional encodings and why ...,with RAG,365,24.8,0.79
3,What are positional encodings and why ...,without RAG,4546,111.2,0.33


## $\color{teal}{\textsf{Observation and Interpretation:}}$

In [20]:
print(f"""
Task 1 - RAG over the transformer paper ({ARXIV_ID}), summary
{"=" * 70}
Document loader   : {loader_used}  -> the source is a PDF, so WebBaseLoader (used in the
                    material notebook for an HTML model card) is not the right tool here.
                    ArxivLoader would also work; we chose the PDF loader because it returns
                    one document PER PAGE, which puts the page number into every citation.
Knowledge source  : {len(data)} pages, {sum(len(d.page_content) for d in data):,} characters,
                    split into {len(docs)} chunks of <= 512 characters (overlap 64).
Embedding / store : {EMBEDDING_MODEL} (768 dimensions) -> FAISS, k = {K} chunks per question.
Generation        : {os_model} via Ollama at {OLLAMA_BASE_URL}, temperature {TEMPERATURE}.

Both questions were answered from the retrieved context alone, and the retrieval did its job: the
self-attention question pulled the definition from page 1, the positional-encoding question pulled
section 3.5 from page 5 - exactly the passages a human would look up.
  1) Self-attention relates different positions of ONE sequence in order to compute a representation
     of that sequence; in the encoder every position attends to all positions of the previous layer,
     in the decoder only up to the current position.
  2) Positional encodings are sine and cosine functions of different frequencies, added to the input
     embeddings. They are needed because the architecture contains NO recurrence and NO convolution,
     so without them the model would have no information about the order of the tokens at all.

COMPARISON WITH THE NO-RAG BASELINE - the table above, and it did not come out as I expected:

* Both variants are factually fine. For a paper this famous that is no surprise: the model has seen it
  many times during training, so this question does NOT show RAG adding missing knowledge.
* What differs is length and shape. With retrieval the answers are ~{len(answer_q1)} and
  ~{len(answer_q2)} characters and answer the question; without retrieval they are around
  {len(no_rag[q1]):,} and {len(no_rag[q2]):,} characters of general tutorial about attention, with
  headings, bullet lists and a history of RNNs nobody asked for.
* And they are far cheaper: {rag_secs[q1]:.0f} s / {rag_secs[q2]:.0f} s with RAG against
  {no_rag_secs[q1]:.0f} s / {no_rag_secs[q2]:.0f} s without - even though the RAG prompt is LONGER.
  The saving is not in reading the context, it is in not generating a thousand tokens of preamble.
* The grounding column is the one that separates them properly: the RAG answers reuse the vocabulary
  of the retrieved chunks, the free-running answers only partly do.

A METHODOLOGICAL NOTE, because it nearly went wrong: my first attempt measured this by counting
technical terms of the paper ("scaled dot-product", "sinusoid", d_model, ...) in each answer. That
metric said the answers WITHOUT retrieval were better grounded - simply because they are ten times
longer and therefore mention more terms in passing. The metric was measuring verbosity. Comparing the
answer against the chunks that were actually retrieved fixes this, and it is the same lesson the
material notebook teaches about retrieval: when a RAG result looks surprising, suspect the measurement
and the retrieval before the model.

So on THIS document RAG buys precision, traceability and speed rather than knowledge. Where it buys
knowledge is visible in task 3: there the same class of model invents biographies for a person it does
not know, and no amount of prompting fixes that - only context does.
""")


Task 1 - RAG over the transformer paper (1706.03762), summary
Document loader   : PyPDFLoader  -> the source is a PDF, so WebBaseLoader (used in the
                    material notebook for an HTML model card) is not the right tool here.
                    ArxivLoader would also work; we chose the PDF loader because it returns
                    one document PER PAGE, which puts the page number into every citation.
Knowledge source  : 15 pages, 39,597 characters,
                    split into 92 chunks of <= 512 characters (overlap 64).
Embedding / store : embeddinggemma (768 dimensions) -> FAISS, k = 5 chunks per question.
Generation        : gemma3 via Ollama at http://localhost:11434, temperature 0.7.

Both questions were answered from the retrieved context alone, and the retrieval did its job: the
self-attention question pulled the definition from page 1, the positional-encoding question pulled
section 3.5 from page 5 - exactly the passages a human would look up.
  1) Self-att